# Protein-based CF

Drugs are users and proteins are items. Predict a drug’s affinity using its training scores for similar proteins.


## Load saved splits

Use the existing training, validation and test pairs.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from utils.protein_cf import (DRUG, PROTEIN, fit_protein_cf, predict_protein_cf,
                             save_protein_cf, load_protein_cf)
from utils.evaluation import mean_squared_error, concordance_index, hit_at_k

PAIR_KEYS = [DRUG, PROTEIN]
MODELS = Path("saved_models")
splits = {
    name: {part: pd.read_csv(Path("data/splits") / f"{name}_{part}.csv",
                           float_precision="round_trip")
           for part in ["train", "val", "test"]}
    for name in ["davis", "kiba"]
}
for name, parts in splits.items():
    train = parts["train"]
    sets = {part: set(frame[PAIR_KEYS].itertuples(index=False, name=None))
            for part, frame in parts.items()}
    assert sets["train"].isdisjoint(sets["val"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["val"].isdisjoint(sets["test"])
    for part, frame in parts.items():
        assert len(sets[part]) == len(frame)
        assert np.isfinite(frame["affinity"]).all()
        for column in PAIR_KEYS:
            assert set(frame[column]) <= set(train[column])
    print(name.upper(), {part: len(frame) for part, frame in parts.items()})


DAVIS {'train': 18040, 'val': 3865, 'test': 3867}
KIBA {'train': 82359, 'val': 17648, 'test': 17650}


## Model

### Drug–protein affinity matrix

Rows are drugs, columns are proteins. Only training observations enter the model. Missing values remain NaN; genuine zero scores remain observed.


In [2]:
models = {name: fit_protein_cf(parts["train"], shrinkage=10.0)
          for name, parts in splits.items()}
for name, model in models.items():
    values = model["values"]
    print(f"{name.upper()}: matrix {values.shape}, observed {np.isfinite(values).mean():.2%}, "
          f"observed zeros {np.count_nonzero(values == 0)}")
    assert np.count_nonzero(np.isfinite(values)) == len(splits[name]["train"])


DAVIS: matrix (68, 379), observed 70.00%, observed zeros 0
KIBA: matrix (2068, 229), observed 17.39%, observed zeros 1


### Protein similarity calculation

Pearson correlation uses only drugs observed for both proteins:

$r_{ij}=\frac{\sum_{u\in C_{ij}}(a_{ui}-\bar a_{i,C})(a_{uj}-\bar a_{j,C})}{\sqrt{\sum_{u\in C_{ij}}(a_{ui}-\bar a_{i,C})^2\sum_{u\in C_{ij}}(a_{uj}-\bar a_{j,C})^2}}$.

Use $s_{ij}=\max(0,r_{ij})n_{ij}/(n_{ij}+10)$. Too few common drugs or constant vectors give zero similarity. Shrinkage reduces small-overlap correlations; missing values never become zeros.


In [3]:
for name, model in models.items():
    assert np.all(np.diag(model["similarities"]) == 0)
    assert np.isfinite(model["similarities"]).all()
    counts = model["common_counts"]
    upper = np.triu_indices(len(counts), 1)
    print(name.upper(), "median shared drugs:", float(np.median(counts[upper])))


DAVIS median shared drugs: 33.0
KIBA median shared drugs: 20.0


### Top-K neighbour selection

Exclude the target itself. Keep positive-similarity proteins observed for the query drug in training, then select the K strongest. Similarity ties use sorted protein IDs.


### Affinity prediction

$\hat a_{ui}=\bar a_i+\frac{\sum_{j\in N_K(u,i)}s_{ij}(a_{uj}-\bar a_j)}{\sum_{j\in N_K(u,i)}s_{ij}}$. Means come from training.

With no suitable neighbour, use the target’s training mean. Unknown targets use the global training mean. No clipping is applied.

Example: target mean 6, neighbour mean 5, drug’s neighbour score 7. With one neighbour, the missing affinity is $6+(7-5)=8$.


### Hyperparameter tuning

Try K = 5, 10, 20, 30 and minimum overlap = 2, 5, 10. Compute similarities once per dataset. Choose lowest validation MSE; ties favour smaller K, then smaller overlap. Test scores are not used.


In [5]:
tuning_rows = []
for name, model in models.items():
    validation = splits[name]["val"]
    for k in [5, 10, 20, 30]:
        for minimum in [2, 5, 10]:
            predictions, fallback = predict_protein_cf(
                model, validation[PAIR_KEYS], k=k, min_common=minimum)
            tuning_rows.append({"Dataset": name.upper(), "K": k,
                                "Min common": minimum,
                                "Val MSE": mean_squared_error(validation["affinity"], predictions),
                                "Val fallback %": 100 * fallback.mean()})
    trials = pd.DataFrame([row for row in tuning_rows if row["Dataset"] == name.upper()])
    best = trials.sort_values(["Val MSE", "K", "Min common"]).iloc[0]
    model.update(k=int(best["K"]), min_common=int(best["Min common"]),
                 dataset=name, score_scale="pKd" if name == "davis" else "KIBA score")
    print(name.upper(), "selected:", best.to_dict())
tuning = pd.DataFrame(tuning_rows)
print(tuning.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


DAVIS selected: {'Dataset': 'DAVIS', 'K': 10, 'Min common': 2, 'Val MSE': 0.2827315495644336, 'Val fallback %': 0.0}
KIBA selected: {'Dataset': 'KIBA', 'K': 5, 'Min common': 2, 'Val MSE': 0.2397717664807622, 'Val fallback %': 0.005666364460562103}
Dataset  K  Min common  Val MSE  Val fallback %
  DAVIS  5           2 0.287675        0.000000
  DAVIS  5           5 0.287675        0.000000
  DAVIS  5          10 0.287675        0.000000
  DAVIS 10           2 0.282732        0.000000
  DAVIS 10           5 0.282732        0.000000
  DAVIS 10          10 0.282732        0.000000
  DAVIS 20           2 0.298709        0.000000
  DAVIS 20           5 0.298709        0.000000
  DAVIS 20          10 0.298709        0.000000
  DAVIS 30           2 0.312811        0.000000
  DAVIS 30           5 0.312811        0.000000
  DAVIS 30          10 0.312811        0.000000
   KIBA  5           2 0.239772        0.005666
   KIBA  5           5 0.240539        0.016999
   KIBA  5          10 0.240325 

## Evaluation

### Evaluation results

Reuse shared MSE, CI and Hit@K functions. Training metrics are in-sample: self-neighbours are excluded, but training scores contribute to means and similarities. Test predictions are evaluated only after selection.

Hit@K uses observed held-out candidates, includes singletons and counts any true-best target as a hit. Prediction ties use target IDs.


In [6]:
result_rows = []
saved_predictions = {}
for name, model in models.items():
    row = {"Dataset": name.upper(), "Model": "Protein CF", "Best K": model["k"],
           "Min common": model["min_common"], "Score scale": model["score_scale"]}
    saved_predictions[name] = {}
    for part_name, part in splits[name].items():
        predictions, fallback = predict_protein_cf(model, part[PAIR_KEYS])
        saved_predictions[name][part_name] = (predictions, fallback)
        label = part_name.capitalize()
        row[f"{label} MSE"] = mean_squared_error(part["affinity"], predictions)
        row[f"{label} CI"] = concordance_index(part["affinity"], predictions)
        row[f"{label} fallback %"] = 100 * fallback.mean()
        if part_name != "train":
            for k in [1, 5]:
                hit = hit_at_k(part, predictions, k)
                row[f"{label} Hit@{k}"] = hit["hit_rate"]
                row[f"{label} eligible ligands"] = hit["eligible_ligands"]
    result_rows.append(row)
results = pd.DataFrame(result_rows)
print(results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Dataset      Model  Best K  Min common Score scale  Train MSE  Train CI  Train fallback %  Val MSE   Val CI  Val fallback %  Val Hit@1  Val eligible ligands  Val Hit@5  Test MSE  Test CI  Test fallback %  Test Hit@1  Test eligible ligands  Test Hit@5
  DAVIS Protein CF      10           2         pKd   0.143402  0.911076          0.000000 0.282732 0.866063        0.000000   0.323529                    68   0.647059  0.289277 0.866785         0.000000    0.382353                     68    0.823529
   KIBA Protein CF       5           2  KIBA score   0.222928  0.833273          0.004857 0.239772 0.826530        0.005666   0.620234                  1967   0.921708  0.254573 0.824670         0.000000    0.590466                   1951    0.894413


### Saving and reloading models

Checkpoints contain training observations, IDs, similarities, overlap counts, means, hyperparameters and scale. Loaded models predict without notebook state.


In [7]:
for name, model in models.items():
    path = MODELS / f"{name}_protein_cf.npz"
    save_protein_cf(model, path)
    loaded = load_protein_cf(path)
    for part_name, part in splits[name].items():
        restored, flags = predict_protein_cf(loaded, part[PAIR_KEYS])
        original, original_flags = saved_predictions[name][part_name]
        np.testing.assert_allclose(restored, original, rtol=1e-12, atol=1e-12)
        np.testing.assert_array_equal(flags, original_flags)
    print(name.upper(), "checkpoint predictions match on all splits")


DAVIS checkpoint predictions match on all splits
KIBA checkpoint predictions match on all splits


### Results and limitations

DAVIS: K = 10, minimum overlap = 2, test MSE = 0.2893, CI = 0.8668, Hit@1 = 0.3824, Hit@5 = 0.8235.

KIBA: K = 5, minimum overlap = 2, test MSE = 0.2546, CI = 0.8247, Hit@1 = 0.5905, Hit@5 = 0.8944.

Test fallback is 0% for both. KIBA uses fallback for one validation pair and four training pairs; DAVIS uses none.

Sparse observations make similarities less reliable. Training metrics are in-sample, and singleton Hit@K always hits. Results cover known drugs and proteins.